In [1]:
import os
import pandas as pd
import numpy as np
import duckdb

### Create and Save DB Tables

In [2]:
FOLDER_PATH = r"database-tables/"

db_tables = {}

for filename in os.listdir(FOLDER_PATH):
    FILE_PATH = os.path.join(FOLDER_PATH, filename)
    table_name = filename.replace(".csv","").replace("-","_")
    print(filename, "-----", table_name)
    db_tables[table_name] = pd.read_csv(FILE_PATH)

agg-batter-vs-phase.csv ----- agg_batter_vs_phase
agg-batter-vs-team.csv ----- agg_batter_vs_team
agg-batter-vs-venue.csv ----- agg_batter_vs_venue
agg-bowler-vs-phase.csv ----- agg_bowler_vs_phase
agg-bowler-vs-team.csv ----- agg_bowler_vs_team
agg-bowler-vs-venue.csv ----- agg_bowler_vs_venue
agg-match.csv ----- agg_match
agg-player-batting.csv ----- agg_player_batting
agg-player-bowling.csv ----- agg_player_bowling
agg-team.csv ----- agg_team
agg-venue.csv ----- agg_venue
dim-match.csv ----- dim_match
dim-player.csv ----- dim_player
dim-season.csv ----- dim_season
dim-team.csv ----- dim_team
dim-umpire.csv ----- dim_umpire
dim-venue.csv ----- dim_venue
fact-delivery-enriched.csv ----- fact_delivery_enriched


C:\Users\darsh\AppData\Local\Temp\ipykernel_17640\1103951499.py:9: DtypeWarning: Columns (0: review_decision) have mixed types. Specify dtype option on import or set low_memory=False.
  db_tables[table_name] = pd.read_csv(FILE_PATH)


fact-delivery.csv ----- fact_delivery


In [3]:
conn = duckdb.connect("database/ipl_analytics.duckdb")

In [4]:
for key,value in db_tables.items():
    try:
        conn.register("temp_df", value)
        conn.execute(f"""
            CREATE OR REPLACE TABLE {key} AS
            SELECT * FROM temp_df
        """)
        print(f"Saved {key} to database.")
    except Exception as e:
        print(f"Error in saving {key} to database - {e}")
    # break

Saved agg_batter_vs_phase to database.
Saved agg_batter_vs_team to database.
Saved agg_batter_vs_venue to database.
Saved agg_bowler_vs_phase to database.
Saved agg_bowler_vs_team to database.
Saved agg_bowler_vs_venue to database.
Saved agg_match to database.
Saved agg_player_batting to database.
Saved agg_player_bowling to database.
Saved agg_team to database.
Saved agg_venue to database.
Saved dim_match to database.
Saved dim_player to database.
Saved dim_season to database.
Saved dim_team to database.
Saved dim_umpire to database.
Saved dim_venue to database.
Saved fact_delivery_enriched to database.
Saved fact_delivery to database.


In [5]:
for key,_ in db_tables.items():

    n_rows = conn.execute(f"""SELECT COUNT(*)FROM {key}""").fetchone()
    print(f"{key} - {n_rows}")


agg_batter_vs_phase - (5857,)
agg_batter_vs_team - (12143,)
agg_batter_vs_venue - (10903,)
agg_bowler_vs_phase - (5187,)
agg_bowler_vs_team - (9493,)
agg_bowler_vs_venue - (8424,)
agg_match - (1169,)
agg_player_batting - (2783,)
agg_player_bowling - (2076,)
agg_team - (156,)
agg_venue - (186,)
dim_match - (1169,)
dim_player - (779,)
dim_season - (18,)
dim_team - (18,)
dim_umpire - (47,)
dim_venue - (37,)
fact_delivery_enriched - (278205,)
fact_delivery - (278205,)


In [6]:
conn.execute("DROP VIEW IF EXISTS temp_df;")
conn.execute("SHOW TABLES").fetchdf()

,name
0,agg_batter_vs_phase
1,agg_batter_vs_team
2,agg_batter_vs_venue
3,agg_bowler_vs_phase
4,agg_bowler_vs_team
5,agg_bowler_vs_venue
6,agg_match
7,agg_player_batting
8,agg_player_bowling
9,agg_player_vs_phase


In [7]:
conn.close()

### Testing the Duckdb Database

In [8]:
from database import execute_query

In [9]:
df = execute_query(query="""
SELECT player_name, season_id, runs, average, strike_rate
FROM agg_player_batting apb
JOIN dim_player dp
ON apb.player_id = dp.player_id
WHERE player_name = 'V KOHLI'
ORDER BY season_id;
""")

df

,player_name,season_id,runs,average,strike_rate
0,V KOHLI,1,165.0,13.75,105.10
1,V KOHLI,2,246.0,18.92,112.33
2,V KOHLI,3,307.0,23.62,144.81
3,V KOHLI,4,557.0,34.81,121.09
4,V KOHLI,5,364.0,24.27,111.66
5,V KOHLI,6,639.0,39.94,139.22
6,V KOHLI,7,359.0,25.64,122.11
7,V KOHLI,8,505.0,31.56,130.83
8,V KOHLI,9,973.0,60.81,152.03
9,V KOHLI,10,308.0,30.80,122.22


In [10]:
df = execute_query(query="""
SELECT player_name, season_id, runs_conceded, wickets, bowling_strike_rate, bowling_average, economy
FROM agg_player_bowling apb
JOIN dim_player dp
ON apb.player_id = dp.player_id
WHERE player_name = 'B KUMAR'
ORDER BY season_id;
""")
df

,player_name,season_id,runs_conceded,wickets,bowling_strike_rate,bowling_average,economy
0,B KUMAR,4,67.0,3.0,22.00,22.33,6.09
1,B KUMAR,5,281.0,8.0,29.25,35.12,7.21
2,B KUMAR,6,371.0,13.0,26.31,28.54,6.51
3,B KUMAR,7,354.0,20.0,15.95,17.70,6.44
4,B KUMAR,8,407.0,18.0,17.22,22.61,7.83
5,B KUMAR,9,490.0,23.0,17.22,21.30,7.42
6,B KUMAR,10,369.0,26.0,12.08,14.19,6.96
7,B KUMAR,11,354.0,9.0,30.78,39.33,7.53
8,B KUMAR,12,461.0,13.0,27.23,35.46,7.81
9,B KUMAR,13,99.0,3.0,28.33,33.00,6.60


In [18]:
df = execute_query(query="""
SELECT player_name, runs, balls, strike_rate
FROM agg_batter_vs_venue apv
JOIN dim_player dp ON apv.player_id = dp.player_id
JOIN dim_venue dv ON apv.venue_id = dv.venue_id         
WHERE dp.player_name = 'MS DHONI' AND dv.venue = 'WANKHEDE STADIUM'
ORDER BY season_id;
""")
df

,player_name,runs,balls,strike_rate
0,MS DHONI,43.0,35.0,122.86
1,MS DHONI,32.0,25.0,128.00
2,MS DHONI,25.0,15.0,166.67
3,MS DHONI,10.0,12.0,83.33
4,MS DHONI,64.0,43.0,148.84
5,MS DHONI,3.0,7.0,42.86
6,MS DHONI,47.0,37.0,127.03
7,MS DHONI,14.0,23.0,60.87
8,MS DHONI,12.0,21.0,57.14
9,MS DHONI,37.0,30.0,123.33


In [12]:
df = execute_query(query="""
SELECT du.umpire, COUNT(fde.umpires_call) as total_umpires_call,
SUM(CASE
    WHEN fde.umpires_call = TRUE THEN 1
    ELSE 0
END) AS right_umpires_call
FROM fact_delivery_enriched fde
JOIN dim_umpire du
ON du.umpire_id = fde.umpire_id
GROUP BY du.umpire
ORDER BY total_umpires_call DESC;
""")
df

,umpire,total_umpires_call,right_umpires_call
0,NITIN MENON,65,13.0
1,VK SHARMA,58,7.0
2,AK CHAUDHARY,57,6.0
3,CB GAFFANEY,52,9.0
4,KN ANANTHAPADMANABHAN,49,5.0
5,UV GANDHE,41,5.0
6,J MADANAGOPAL,36,5.0
7,YC BARDE,32,6.0
8,MV SAIDHARSHAN KUMAR,28,7.0
9,A NAND KISHORE,28,3.0


In [13]:
df = execute_query("""
SELECT dp.player_name, apb.wickets, apb.bowling_strike_rate, apb.bowling_average, apb.economy, apb.best_bowling_figures, ds.season
FROM agg_player_bowling apb
JOIN dim_player dp
ON apb.player_id = dp.player_id
JOIN dim_season ds
ON apb.season_id = ds.season_id
WHERE dp.player_name.contains('BUMRAH');
""")
df

,player_name,wickets,bowling_strike_rate,bowling_average,economy,best_bowling_figures,season
0,JJ BUMRAH,3.0,14.00,23.33,10.00,3/32,2013.0
1,JJ BUMRAH,5.0,47.60,60.20,7.52,2/22,2014.0
2,JJ BUMRAH,3.0,30.00,61.33,12.27,1/38,2015.0
3,JJ BUMRAH,15.0,20.80,27.07,7.81,3/13,2016.0
4,JJ BUMRAH,20.0,18.10,22.20,7.28,3/7,2017.0
5,JJ BUMRAH,17.0,19.06,21.88,6.89,3/15,2018.0
6,JJ BUMRAH,20.0,18.70,20.85,6.62,3/20,2019.0
7,JJ BUMRAH,29.0,12.83,14.48,6.77,5/29,2020.0
8,JJ BUMRAH,21.0,15.71,19.52,7.45,3/36,2021.0
9,JJ BUMRAH,15.0,21.33,25.53,7.09,5/10,2022.0


In [17]:
df = execute_query("""
SELECT player_name, matches, season, runs, balls, average, strike_rate, highest_score
FROM agg_batter_vs_team apt
JOIN dim_player dp
ON apt.player_id = dp.player_id
JOIN dim_season ds
ON apt.season_id = ds.season_id
JOIN dim_team dt
ON apt.opposition_team_id = dt.team_id
WHERE dp.player_name='MS DHONI' AND dt.team_name='MUMBAI INDIANS';
""")
df

,player_name,matches,season,runs,balls,average,strike_rate,highest_score
0,MS DHONI,2,2008.0,73.0,51.0,36.50,143.14,43.0
1,MS DHONI,2,2009.0,59.0,48.0,29.50,122.92,36.0
2,MS DHONI,2,2010.0,53.0,33.0,26.50,160.61,31.0
3,MS DHONI,1,2011.0,3.0,6.0,3.00,50.00,3.0
4,MS DHONI,3,2012.0,80.0,41.0,26.67,195.12,51.0
5,MS DHONI,3,2013.0,124.0,83.0,41.33,149.40,63.0
6,MS DHONI,2,2014.0,36.0,23.0,18.00,156.52,22.0
7,MS DHONI,4,2015.0,60.0,52.0,15.00,115.38,39.0
8,MS DHONI,1,2016.0,24.0,24.0,24.00,100.00,24.0
9,MS DHONI,4,2017.0,69.0,62.0,17.25,111.29,40.0
